In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import math
import ndlib.models.ModelConfig as mc
import ndlib.models.epidemics as ep
from ndlib.utils import multi_runs
import networkx as nx

# Problem 1: Gillespie’s Direct Algorithm and Stochastic Hallmarks

### Stochastic Model Functions

In [ ]:
def process_probabilities(state, params):
    '''
        Calculates the Propensity Functions for the SIR Model 
        
        Parameters :
        
        state (dict): A dictionary containing the current counts of each compartment:
            - 'X' (int) : Number of susceptible individuals
            - 'Y' (int) : Number of infected individuals
            - 'Z' (int) : Number of recovered individuals
        params (dict): A dictionary containing the model parameters:
            - 'beta' (float) : Transmission rate
            - 'gamma' (float) : Recovery rate
            - 'mu' (float) : Birth and natural death rate

        Returns :
        a(np.ndarray): An array consisting of Propensity Functions, wherein 
        a[0] : Transmission Propensity 
        a[1] : Recovery Propensity 
        a[2] : Birth Propensity 
        a[3] : Death Propensity from Susceptible Population (X)
        a[4] : Death Propensity from Infected Popultion (Y)
        a[5] : Death Propensity from Recovered Population (Z)
        
    '''
    # Initialize variables
    X, Y, Z = state['X'], state['Y'], state['Z']
    a = np.zeros(6)
    N = X + Y + Z

    # Calculate the rates
    a[0] = (params['beta'] * X * Y) / N # Transmission rate
    a[1] = params['gamma'] * Y # Recovery Rates
    a[2] = params['mu'] * N # Birth
    a[3] = params['mu'] * X # Death from X
    a[4] = params['mu'] * Y # Death from Y
    a[5] = params['mu'] * Z # Death from Z
    
    return a

In [ ]:
def SIR_stochastic(params, initial_values, end_time):
    '''
     Simulate the SIR epidemic model using a stochastic approach.
    
        Parameters :
        
            params (dict) : A dictionary containing the model parameters:
                - 'beta' (float) : The transmission rate of the disease
                - 'gamma' (float) : The recovery rate of the infected individuals
                - 'mu' (float) : The birth and natural death rate of the population
        
            initial_values (dict) : A dictionary containing the initial state of the system:
                - 'X' (int) : Initial number of susceptible individuals
                - 'Y' (int) : Initial number of infected individuals
                - 'Z' (int) : Initial number of recovered individuals
        
            end_time (float) : The time until which the simulation will run.

        Returns:
            dict: A dictionary containing the results of the simulation:
                - 'time' (list) : List of time points when events occurred.
                - 'X' (list) : List of the number of susceptible individuals at each point in time.
                - 'Y' (list): List of the number of infected individuals at each point in time.
                - 'Z' (list): List of the number of recovered individuals at each point in time.
                - 'extinction' (boolean): Specifies whether the simulation ended in an extinction or not.
                - 'extinction_time' (float): The time point when the disease got extinct. 
    '''
  
    state = initial_values.copy() # Copying and storing the initial population of susceptible, infected and recovered individuals 
    time = 0
    result = {'time': [], 'X': [], 'Y': [], 'Z': [], 'extinction': False, 'extinction_time': float('inf')}

    # Run the algorithm until we get to the time wanted or until there is an extinction of the disease
    while time < end_time and state['Y'] > 0:        
        # Get the rates
        probs = process_probabilities(state, params)

        # Get the time until the next event
        tau = (- 1 / sum(probs)) * np.log(np.random.random())
        time += tau

        # Get the type of the event
        which = process(probs)

        # Update the state
        new_state = update_state(state, which)

        # Append the time and new state to the previous results
        result['time'].append(time)
        result['X'].append(new_state['X'])
        result['Y'].append(new_state['Y'])
        result['Z'].append(new_state['Z'])    

    if state['Y'] < 1:
        result['extinction'] = True
        result['extinction_time'] = time
        
    return result

In [ ]:
def process(probs):
    '''
    Function selects next event based on propensities and identifies which event should occur based on the cumulative probability distribution 
    
    Parameters :
        probs (numpy.ndarray or list) : A list or array of propensity functions (rates) for each possible event

    Returns :
        int : The index of the event that occurs
    '''
    # Initialize variables
    which = 0
    cumulative_sum = probs[0]

    # Calculate r2
    r2 = np.random.random() * sum(probs)

    # Find which event occured
    while r2 > cumulative_sum:
        which += 1
        cumulative_sum += probs[which]
        
    return which + 1

In [ ]:
def update_state(state, which):
    '''
    Updates the state of the population based on the event that occur 
        
        Parameters:
            state (dict) : The current state of the population
            which (int) : The event index 
        
        Returns:
            dict : The updated state of the population with modified values for 'X', 'Y', and 'Z'.
    '''
    if which == 1: # Infection
        state['X'] -= 1
        state['Y'] += 1
    elif which == 2: # Recovery
        state['Y'] -= 1
        state['Z'] += 1
    elif which == 3: # Birth
        state['X'] += 1 
    elif which == 4: # Death from X
        state['X'] -= 1
    elif which == 5: # Death from Y
        state['Y'] -= 1
    elif which == 6: # Death from Z
        state['Z'] -= 1

    return state

In [ ]:
def run_multiple_simulations(params, initial_values, end_time, no_runs):
    '''
        Function to run the SIR Stochastic Simulation Multiple times 

        Parameters :
            params (dict): Dictionary containing Model Parameters 
            initial_values (dict): Initial values of the population
            end_time(int): Time until which simulation runs 
            no_runs(int): Number of times the simulation should be run 

        Returns :
            results(List) : contains simulation result for each run 
    '''
    results = [{} for _ in range(no_runs)]
    
    for i in range(no_runs):
        results[i] = SIR_stochastic(params, initial_values, end_time)
        
    return results

### Functions for plotting the results

In [ ]:
def initialize_plot():
    '''
        Initializes Plot dimensions and sets DPI of the plot to 300 
    '''
    plt.figure(figsize=(10, 6))
    plt.rcParams['figure.dpi'] = 300

In [ ]:
def plot_single_result(result):
    time = result['time']
    plt.plot(time, result['X'], color='blue', label = 'Susceptible', linestyle = '-')
    plt.plot(time, result['Y'], color='red', label = 'Infected', linestyle = '-')
    plt.plot(time, result['Z'], color='green', label = 'Recovered', linestyle = '-')

In [ ]:
def add_description_to_plot():
    plt.xlabel('Time in days')
    plt.ylabel('Population')
    plt.title('SIR Model Simulation')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

In [ ]:
def plot_multiple_results(results):
    # Plot the first one and include labels
    first_result = results[0]
    time = first_result['time']
    
    plt.plot(time, first_result['X'], color='blue', alpha=0.1, label = 'Susceptible')
    plt.plot(time, first_result['Y'], color='red', alpha=0.1, label = 'Infected')
    plt.plot(time, first_result['Z'], color='green', alpha=0.1, label = 'Recovered')

    # Plot the rest (we assume there is more than 1 result)
    for result in results[1:]:
        time = result['time']
        
        plt.plot(time, result['X'], color='blue', alpha=0.1)
        plt.plot(time, result['Y'], color='red', alpha=0.1)
        plt.plot(time, result['Z'], color='green', alpha=0.1)

In [ ]:
def add_description_to_plot_multiple_simulations():
    plt.xlabel('Time in days')
    plt.ylabel('Population')
    plt.title('SIR Model Simulation')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

In [ ]:
def show_plot():
    plt.show()

In [ ]:
def save_plot(file_name):
    plt.savefig(rf'{file_name}.png', dpi=300)

In [ ]:
def get_initial_values_dict(N):
    return {'X': N - 1, 'Y': 1, 'Z': 0}

### Testing

In [ ]:
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

N = 1e3
initial_values = get_initial_values_dict(N)

end_time = 50

result = SIR_stochastic(params, initial_values, end_time)

In [ ]:
initialize_plot()
plot_single_result(result)
add_description_to_plot()
show_plot()

In [ ]:
# TEST run multiple simulations
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

N = 1e3
initial_values = get_initial_values_dict(N)

end_time = 50
no_runs = 50

results = run_multiple_simulations(params, initial_values, end_time, no_runs)

In [ ]:
# TEST plot multiple simulations
initialize_plot()
plot_multiple_results(results)
add_description_to_plot_multiple_simulations()
show_plot()

### Deterministic SIR Model 

In [ ]:
def SIR_deterministic(y, t, beta, gamma, mu):
    '''
        Function the returns the set of Ordinary Differential Equations that govern the evolution of Susceptible , Infected , Recovered population 
        Parameters:
        y (np.ndarray): 
            Array containing the current state variables [X, Y, Z], where:
            - X: Number of susceptible individuals.
            - Y: Number of infected individuals.
            - Z: Number of recovered individuals.

        t (float): 
            The current time point 

        beta (float): 
            Transmission rate of the infection.

        gamma (float): 
            Recovery rate of infected individuals.

        mu (float): 
            Natural Birth and death rate.
    '''

    X, Y, Z = y
    N = X + Y + Z

    # Define the equations
    dXdt = - ((beta * X * Y )/ N) + (mu * N) - (mu * X)
    dYdt = ((beta * X * Y) / N) - (gamma * Y) - (mu * Y)
    dZdt = (gamma * Y) - (mu * Z)
    
    return [dXdt, dYdt, dZdt]

In [ ]:
def solve_SIR_deterministic(params, initial_values, end_time):
    '''
        Function to integrate the system of ODE's over a period of time using initial values and parameters 
        
            Parameters:
                params (dict) : A dictionary containing model parameters:
        
                initial_values (dict) : A dictionary containing the initial population values
        
                end_time (float) : The time up to which the system should be simulated.
        
            Returns:
                solutiont(np.ndarray) : 2D array where each row contains the solution [X, Y, Z] for each time point in the integration.
    '''
    # Prepare time points
    time_points = np.linspace(0, end_time, 100)
    initial_values_array = [initial_values['X'], initial_values['Y'], initial_values['Z']]

    # Run solver
    solution = odeint(SIR_deterministic, initial_values_array, time_points, args=(params['beta'], params['gamma'], params['mu']))

    return solution

In [ ]:
# TEST solve deterministic SIR
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

N = 1e3
initial_values = get_initial_values_dict(N)

end_time = 50

deterministic_result = solve_SIR_deterministic(params, initial_values, end_time)

In [ ]:
# TEST plot deterministic SIR
deterministic_result_dict = {'time': np.linspace(0, end_time, 100), 'X': deterministic_result[:, 0], 'Y': deterministic_result[:, 1], 'Z': deterministic_result[:, 2]}

initialize_plot()
plot_single_result(deterministic_result_dict)
add_description_to_plot()
# save_plot('testin_single_plot')
show_plot()

In [ ]:
# TEST plot deterministic + multiple simulations
deterministic_result_dict = {'time': np.linspace(0, end_time, 100), 'X': deterministic_result[:, 0], 'Y': deterministic_result[:, 1], 'Z': deterministic_result[:, 2]}

initialize_plot()
plot_single_result(deterministic_result_dict)
plot_multiple_results(results)
add_description_to_plot()
# save_plot('testin_single_plot')
show_plot()

In [ ]:
# TEST run multiple simulations + deterministic
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

N = 1e3
initial_values = get_initial_values_dict(N)

end_time = 50
no_runs = 50

# Solve
results = run_multiple_simulations(params, initial_values, end_time, no_runs)
deterministic_result = solve_SIR_deterministic(params, initial_values, end_time)

# Plot
deterministic_result_dict = {'time': np.linspace(0, end_time, 100), 'X': deterministic_result[:, 0], 'Y': deterministic_result[:, 1], 'Z': deterministic_result[:, 2]}

initialize_plot()
plot_single_result(deterministic_result_dict)
plot_multiple_results(results)
add_description_to_plot()
show_plot()

## Problem 1.2 Variance, covariance, mean

In [ ]:
def get_variance_covariance_mean(params, N, end_time, no_runs):
    '''
    Function to run the SIR Stochastic Simulation Multiple times and calculate statistics

    Parameters :
        params (dict): Dictionary containing Model Parameters 
        N: The population size
        end_time(int): Time until which simulation runs 
        no_runs(int): Number of times the simulation should be run 

     Returns:
        dict: A dictionary containing the following keys and their corresponding values:
            - 'time_grid' (ndarray): The grid of time points at which the statistics are computed.
            - 'mean_X' (ndarray): The mean value of the Susceptible population at each time point.
            - 'mean_Y' (ndarray): The mean value of the Infected population at each time point.
            - 'mean_Z' (ndarray): The mean value of the Recovered population at each time point.
            - 'var_X' (ndarray): The variance of the Susceptible population at each time point.
            - 'var_Y' (ndarray): The variance of the Infected population at each time point.
            - 'var_Z' (ndarray): The variance of the Recovered population at each time point.
            - 'cov_XY' (ndarray): The covariance between the Susceptible and Infected populations at each time point.
    '''
    # Get initial values
    initial_values = get_initial_values_dict(N)

    # Prepare time grid for interpolation
    num_points = 50
    time_grid = np.linspace(0, end_time, num_points)

    # Initialize lists to store interpolated results for each run
    X_values, Y_values, Z_values = [], [], []

    # Run the simulations
    results = run_multiple_simulations(params, initial_values, end_time, no_runs)

    # Get interpolated values for points in time grid
    for result in results:
        # Remove the results where the population got extinct after only a few time steps
        if result['extinction']:
            continue

        # Add the interpolated values
        X_values.append(np.interp(time_grid, result['time'], result['X']))
        Y_values.append(np.interp(time_grid, result['time'], result['Y']))
        Z_values.append(np.interp(time_grid, result['time'], result['Z']))

    # Convert lists to numpy arrays for easier calculations (shape: [run_count, num_points])
    X_values = np.array(X_values)
    Y_values = np.array(Y_values)
    Z_values = np.array(Z_values)

    # Calculate the mean at each time point (over all runs)
    mean_X = np.mean(X_values, axis=0)
    mean_Y = np.mean(Y_values, axis=0)
    mean_Z = np.mean(Z_values, axis=0)

    # Calculate the variance at each time point (over all runs)
    var_X = np.var(X_values, axis=0)
    var_Y = np.var(Y_values, axis=0)
    var_Z = np.var(Z_values, axis=0)

    # Calculate the covariance between S and I at each time point (over all runs)
    cov_XY = np.mean((X_values - mean_X) * (Y_values - mean_Y), axis=0)

    # Return the mean, variance, and covariance results
    return {
        'time_grid': time_grid,
        'mean_X': mean_X,
        'mean_Y': mean_Y,
        'mean_Z': mean_Z,
        'var_X': var_X,
        'var_Y': var_Y,
        'var_Z': var_Z,
        'cov_XY': cov_XY
    }

In [ ]:
def get_mean_Y_from_results(results, end_time, num_points):
    '''
    Calculate the mean of the Infected population from results.
    
    Parameters:
        results (list of dict): A list of dictionaries containing simulation results.
        end_time (int): The total time until which the simulation results are evaluated.
        num_points (int): The number of points in the time grid for interpolation.

    Returns:
        dict: A dictionary containing:
            - 'time_grid' (ndarray): The grid of time points at which the mean is computed.
            - 'mean_Y' (ndarray): The mean value of the Infected population at each time point.
    '''
     # Prepare time grid for interpolation
    time_grid = np.linspace(0, end_time, num_points)

    # Initialize lists to store interpolated results for each run
    Y_values = []
    
    # Get interpolated values for points in time grid
    for result in results:
        time_points = result['time']
        infecteds = result['Y']
        
        # If there is an extinction, fill in the rest of the values with
        if result['extinction']:
            continue

        # Add the interpolated values
        Y_values.append(np.interp(time_grid, time_points, infecteds))

    # Convert lists to numpy arrays for easier calculations (shape: [run_count, num_points])
    Y_values = np.array(Y_values)

    # Calculate the mean at each time point (over all runs)
    mean_Y = np.mean(Y_values, axis=0)

    return {'time_grid': time_grid, 'mean_Y': mean_Y}

In [ ]:
# TEST get_variance_covariance_mean
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

N = 1e3

end_time = 50
no_runs = 50

statistics = get_variance_covariance_mean(params, N, end_time, no_runs)

In [ ]:
# TEST plot
time_grid = np.linspace(0, end_time, 50)
statistics_dict = {'time': time_grid, 'X': statistics['var_X'], 'Y': statistics['var_Y'], 'Z': statistics['var_Z']}

initialize_plot()
plot_single_result(statistics_dict)
add_description_to_plot()
show_plot()

In [ ]:
# TEST variance vs mean
initialize_plot()

plt.plot(time_grid, statistics['var_Y'], color='blue', label = 'Variance (I)', linestyle = '-')
plt.plot(time_grid, [mean * 100 for mean in statistics['mean_Y']], color='red', label = 'Mean (I)', linestyle = '-')

plt.legend()

show_plot()

In [ ]:
# TEST for different values of N
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

end_time = 50
no_runs = 50

statistics_1e2 = get_variance_covariance_mean(params, 1e2, end_time, no_runs)
statistics_1e3 = get_variance_covariance_mean(params, 1e3, end_time, no_runs)
statistics_1e4 = get_variance_covariance_mean(params, 1e4, end_time, no_runs)

In [ ]:
# TEST plot the different variances
initialize_plot()

plt.plot(time_grid, [var/1e2 for var in statistics_1e2['var_Y']], color='blue', label = 'Variance (I), N = 1e2', linestyle = '-')
plt.plot(time_grid, [var/1e3 for var in statistics_1e3['var_Y']], color='red', label = 'Variance (I), N = 1e3', linestyle = '-')
plt.plot(time_grid, [var/1e4 for var in statistics_1e4['var_Y']], color='green', label = 'Variance (I), N = 1e4', linestyle = '-')

plt.legend()

show_plot()

In [ ]:
# TODO: Clean up
# Show how the mean changes versus the deterministic mean
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

end_time = 50
no_runs = 50
start_time = 50
N = 1e3
initial_values = get_initial_values_dict(N)

# Solve
results = run_multiple_simulations(params, initial_values, end_time, no_runs)
deterministic_result = solve_SIR_deterministic(params, initial_values, end_time)

num_points = end_time
mean_Y = get_mean_Y_from_results(results, end_time, num_points)

deterministic_result_dict = {'time': np.linspace(0, end_time, 100), 'X': deterministic_result[:, 0], 'Y': deterministic_result[:, 1], 'Z': deterministic_result[:, 2]}

# Plot
initialize_plot()

# Plot deterministic Y
plt.plot(deterministic_result_dict['time'], deterministic_result_dict['Y'], color='red', label = 'Deterministic', linestyle = '-')

# Plot stochastic mean of Y
plt.plot(mean_Y['time_grid'], mean_Y['mean_Y'], color='blue', label = 'Mean stochastic', linestyle = '-')

plt.axis([0, end_time, 0, 700])
plt.legend()
plt.xlabel('Time in days')
plt.ylabel('Number infected')
plt.title(f'Number of infected people deterministic vs stochastic for t > {start_time}')
show_plot()

## Problem 1.3 Stochastic Resonance and Increased Transients

In [ ]:
def combine_stochastic_results(stochastic_results):
    result = {'time': [], 'X': [], 'Y': [], 'Z': []}
    for out in stochastic_results:
        result['time'].extend(out['time'])
        result['X'].extend(out['X'])
        result['Y'].extend(out['Y'])
        result['Z'].extend(out['Z'])
    return result

In [ ]:
def get_unique_time_values(result):
    '''
        Function to get unique time points from the simulations 
    '''
    time, unique_indices = np.unique(result['time'], return_index=True)
    X = np.array(result['X'])[unique_indices]
    Y = np.array(result['Y'])[unique_indices]
    Z = np.array(result['Z'])[unique_indices]
    return time, X, Y, Z

In [ ]:
def calculate_means(time, X, Y, Z, N):
    meanTime, meanS, meanI, meanR = [], [], [], []
    for i in range(0, len(time) - N + 1, N):
        meanTime.append(np.mean(time[i:i+N]))
        meanS.append(np.mean(X[i:i+N]))
        meanI.append(np.mean(Y[i:i+N]))
        meanR.append(np.mean(Z[i:i+N]))
    return meanTime, meanS, meanI, meanR

In [ ]:
def calculate_transients(meanTime, meanI, time, det_results, N):
    '''
        The function computes the transient deviations (differences) between the mean infected population from the stochastic SIR simulations (`meanI`) and the infected population from the deterministic SIR model (`det_I`). The transient deviations are normalized by the stochastic infected population.
        
        Parameters:
        meanTime (List):
            The time points at which the mean values of the stochastic simulations are calculated.
        meanI (List) :
            The mean values of the infected population (I) obtained from the stochastic SIR model.
        time (List):
            The time points corresponding to the deterministic model results.
        det_results (List):
            The solution of the deterministic SIR model, where the second column represents the infected population (I).
        N (int) :
            The chunk size or step size used to group data for averaging purposes.
        
        Returns:
        transients (List):
            A list of the relative transient deviations between the stochastic and deterministic infected populations, normalized by the stochastic infected population.

    '''
    transients = []
    for i in range(0, len(meanTime)):
        total_stochastic = meanI[i]  # Stochastic I only
        det_I = np.interp(meanTime[i], time, det_results[:, 1])  # Interpolating deterministic I

        # Calculate transient deviation as a ratio of the total population
        transient = np.sqrt((meanI[i] - det_I) ** 2)
        relative_transient = transient / total_stochastic if total_stochastic != 0 else 0  # Avoid division by zero
        transients.append(relative_transient)
    return transients

In [ ]:
def plot_transients(meanTime, transients):
    '''
    Function plots the Relative Transient Deviation  from Deterministic Equilibrium  
    '''
    initialize_plot()
    plt.plot(meanTime, transients, label='Relative Transient Deviations', linestyle='-')
    plt.xlabel('Time')
    plt.ylabel('Transient Deviation (Normalized by Population)')
    plt.title('Relative Transient Deviations from Deterministic Equilibrium')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    show_plot()

In [ ]:
def process_stochastic_results(stochastic_results, time_points, det_results, initial):
    '''
    Function combines the results from multiple stochastic runs, calculates the mean values of susceptible, infected, and recovered populations, and computes
    the transient deviations between the stochastic and deterministic solutions
    Parameters:
        stochastic_results (list of dict):
            List where each element is a dictionary containing time points and corresponding 
            population values (X, Y, Z) from individual stochastic simulation runs

        time_points (np.ndarray):
            NumPy array of time points corresponding to the deterministic solution

        det_results (np.ndarray):
            NumPy array where each row contains the deterministic solution for susceptible (X),
            infected (Y), and recovered (Z) populations at different time points

    '''
    
    # Combine results from all stochastic runs
    combined_results = combine_stochastic_results(stochastic_results)

    # Get unique time points and corresponding values
    time, X, Y, Z = get_unique_time_values(combined_results)

    # Calculate mean values
    N_chunk = math.ceil(sum(initial.values()) / 100)  # Adjust chunk size based on initial values
    meanTime, meanS, meanI, meanR = calculate_means(time, X, Y, Z, N_chunk)

    # Calculate transients
    transients = calculate_transients(meanTime, meanI, time_points, det_results, N_chunk)

    print(f"Average relative transient: {np.mean(transients)}")
    plot_transients(meanTime, transients)

In [ ]:
def SIR_stoch_run_with_transients(beta=1, X=999):
    '''
        Runs the SIR stochastic model multiple times and compares the results with the deterministic model to calculate transient deviations.
        Parameters:
        beta (float): optional parameter 
            The transmission rate parameter for the disease. Default is set to 1 .
        X (int): optional parameter
            The initial number of susceptible individuals (S) in the population. Default is set to 999 .
    '''
    params = {
        'beta': beta,  
        'gamma': 0.1,
        'mu': 0.02 / 365
    }
    initial = {
        'X': X,     
        'Y': 1,     
        'Z': 0      
    }
    
    end_time = 50  # end of simulation time span starting at 0
    run_count = 50 # number of runs

    # Solve the deterministic model and retrieve the solution
    det_results = solve_SIR_deterministic(params, initial, end_time)
    time_points = np.linspace(0, end_time, 100)

    # Run stochastic simulations
    stochastic_results = run_multiple_simulations(params, initial, end_time, run_count)

    # Process stochastic results to calculate transients
    process_stochastic_results(stochastic_results, time_points, det_results, initial)

In [ ]:
SIR_stoch_run_with_transients()

### Problem 1.3 Stochastic Resonance

In [ ]:
# TODO: Clean up
# Run multiple simulations for longer time and do not include the beginning of the epidemics
# Test with parameter values that generate oscillations
beta = 0.9
gamma = 1/3
mu = 1/60
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

N = 1e4
initial_values = get_initial_values_dict(N)

end_time = 200
no_runs = 50

# Solve
results = run_multiple_simulations(params, initial_values, end_time, no_runs)
deterministic_result = solve_SIR_deterministic(params, initial_values, end_time)

In [ ]:
# Plot
deterministic_result_dict = {'time': np.linspace(0, end_time, 100), 'X': deterministic_result[:, 0], 'Y': deterministic_result[:, 1], 'Z': deterministic_result[:, 2]}

start_time = 0
num_points = end_time

mean_Y = get_mean_Y_from_results(results, end_time, num_points)

initialize_plot()

# Plot deterministic Y
plt.plot(deterministic_result_dict['time'], deterministic_result_dict['Y'], color='red', label = 'Deterministic', linestyle = '-')

# Plot stochastic mean of Y
plt.plot(mean_Y['time_grid'], mean_Y['mean_Y'], color='blue', label = 'Mean stochastic', linestyle = '-')

# Plot stochastic Y
for result in results:
    plt.plot(result['time'], result['Y'], color='red', alpha=0.1)

plt.axis([start_time, end_time, 0, np.max(deterministic_result[:, 1]) + 100])
plt.legend()
plt.xlabel('Time in days')
plt.ylabel('Number infected')
plt.title(f'Number of infected people deterministic vs stochastic for t > {start_time}')
show_plot()

In [ ]:
def run_simulation(beta, N, end_time, no_runs):
    params = {'beta': beta, 'gamma': 1/3, 'mu': 1/60}  # Assuming gamma and mu are constants
    initial_values = get_initial_values_dict(N)  # Assuming this function is defined elsewhere
    results = run_multiple_simulations(params, initial_values, end_time, no_runs)
    return results

def plot_results(results, deterministic_result, end_time):
    deterministic_result_dict = {'time': np.linspace(0, end_time, 100), 
                                  'X': deterministic_result[:, 0], 
                                  'Y': deterministic_result[:, 1], 
                                  'Z': deterministic_result[:, 2]}

    start_time = 0
    num_points = end_time
    mean_Y = get_mean_Y_from_results(results, end_time, num_points)  # Assuming this is defined elsewhere

    initialize_plot()

    # Plot deterministic Y
    plt.plot(deterministic_result_dict['time'], deterministic_result_dict['Y'], 
             color='red', label='Deterministic', linestyle='-')

    # Plot stochastic mean of Y
    plt.plot(mean_Y['time_grid'], mean_Y['mean_Y'], color='blue', label='Mean stochastic', linestyle='-')

    # Plot stochastic Y
    for result in results:
        plt.plot(result['time'], result['Y'], color='red', alpha=0.1)

    plt.axis([start_time, end_time, 0, np.max(deterministic_result[:, 1]) + 100])
    plt.legend()
    plt.xlabel('Time in days')
    plt.ylabel('Number infected')
    plt.title(f'Number of infected people deterministic vs stochastic for t > {start_time}')
    show_plot()

def vary_beta(N, end_time, no_runs, beta_values):
    for beta in beta_values:
        print(f"Running simulation with beta: {beta}")
        results = run_simulation(beta, N, end_time, no_runs)
        deterministic_result = solve_SIR_deterministic({'beta': beta, 'gamma': 1/3, 'mu': 1/60}, 
                                                        get_initial_values_dict(N), end_time)
        plot_results(results, deterministic_result, end_time)

def vary_N(beta, end_time, no_runs, N_values):
    for N in N_values:
        results = run_simulation(beta, N, end_time, no_runs)
        deterministic_result = solve_SIR_deterministic({'beta': beta, 'gamma': 1/3, 'mu': 1/60},get_initial_values_dict(N), end_time)
        plot_results(results, deterministic_result, end_time)

# Example usage
end_time = 200
no_runs = 50

# Varying beta
beta_values = [0.5, 0.7, 0.9]
vary_beta(1e4, end_time, no_runs, beta_values)

# Varying N
N_values = [1e3, 1e4, 1e5]
vary_N(0.9, end_time, no_runs, N_values)


In [ ]:
def run_simulation(beta, N, end_time, no_runs):
    params = {'beta': beta, 'gamma': 1/3, 'mu': 1/50}  # Assuming gamma and mu are constants
    initial_values = get_initial_values_dict(N)  # Assuming this function is defined elsewhere
    results = run_multiple_simulations(params, initial_values, end_time, no_runs)
    return results

def plot_results(results, deterministic_result, end_time):
    deterministic_result_dict = {'time': np.linspace(0, end_time, 100), 
                                  'X': deterministic_result[:, 0], 
                                  'Y': deterministic_result[:, 1], 
                                  'Z': deterministic_result[:, 2]}

    start_time = 0
    num_points = end_time
    mean_Y = get_mean_Y_from_results(results, end_time, num_points)  # Assuming this is defined elsewhere

    initialize_plot()

    # Plot deterministic Y
    plt.plot(deterministic_result_dict['time'], deterministic_result_dict['Y'], 
             color='red', label='Deterministic', linestyle='-')

    # Plot stochastic mean of Y
    plt.plot(mean_Y['time_grid'], mean_Y['mean_Y'], color='blue', label='Mean stochastic', linestyle='-')

    # Plot stochastic Y
    for result in results:
        plt.plot(result['time'], result['Y'], color='red', alpha=0.1)

    plt.axis([start_time, end_time, 0, np.max(deterministic_result[:, 1]) + 100])
    plt.legend()
    plt.xlabel('Time in days')
    plt.ylabel('Number infected')
    plt.title(f'Number of infected people deterministic vs stochastic for t > {start_time}')
    show_plot()

def vary_beta(N, end_time, no_runs, beta_values):
    for beta in beta_values:
        print(f"Running simulation with beta: {beta}")
        results = run_simulation(beta, N, end_time, no_runs)
        deterministic_result = solve_SIR_deterministic({'beta': beta, 'gamma': 1/3, 'mu': 1/60}, 
                                                        get_initial_values_dict(N), end_time)
        plot_results(results, deterministic_result, end_time)

def vary_N(beta, end_time, no_runs, N_values):
    for N in N_values:
        results = run_simulation(beta, N, end_time, no_runs)
        deterministic_result = solve_SIR_deterministic({'beta': beta, 'gamma': 1/3, 'mu': 1/60},get_initial_values_dict(N), end_time)
        plot_results(results, deterministic_result, end_time)

# Example usage
end_time = 400
no_runs = 50

# Varying beta
beta_values = [1, 1.25,1.5]
vary_beta(1e3, end_time, no_runs, beta_values)

# Varying N
#N_values = [1e3, 1e4, 1e5]
#vary_N(1.25, end_time, no_runs, N_values)


## Problem 1.4 Extinction events and Critical Community Size

In [ ]:
def get_number_of_extinctions(N, params, end_time, no_runs):
    initial_values = get_initial_values_dict(N)

    results = run_multiple_simulations(params, initial_values, end_time, no_runs)
    
    no_extinctions = 0
    
    for result in results:
        if result['extinction'] == True:
            no_extinctions += 1

    return no_extinctions

In [ ]:
# TEST get_number_of_extinctions and save in matrix to be able to plot it
betas = np.linspace(0.1, 2, 20)
gamma = 0.1
mu = 0.02/365

N = [1e0, 1e1, 1e2, 1e3] # Note: you can add 1e4 as well, but this will take more time to generate

end_time = 50
no_runs = 100

extinctions = np.zeros((len(betas), len(N)))

for i, beta in enumerate(betas):
    params = {'beta': beta, 'gamma': gamma, 'mu': mu}
    R_0 = beta/gamma   

    for j, n in enumerate(N): 
        no_extinctions = get_number_of_extinctions(n, params, end_time, no_runs)

        extinctions[i][j] = no_extinctions

In [ ]:
# Convert N to a logarithmic scale for plotting
log_N = np.log10(N)

# Get R_0 frome betas
gamma = 0.1
R_0 = [beta/gamma for beta in betas]

# Create a 2D grid of betas and log_N for plotting
R_0_mesh, LogN = np.meshgrid(R_0, log_N)

# Plotting the 3D surface plot
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
ax.plot_surface(R_0_mesh, LogN, extinctions.T, cmap='viridis')  # Transpose the extinction matrix

ax.set_xlabel('R_0')
ax.set_ylabel('Population (log10)')
ax.set_zlabel('Number of Extinctions')
ax.set_title('Extinctions vs R_0 and Population Size')

ax.set_ylim(max(log_N), min(log_N)) # Reverse N to go from large to small

plt.tight_layout()
plt.show()

### Testing extinction with time

In [ ]:
def get_number_of_extinctions_time_series(N, params, end_time, no_runs):
    initial_values = get_initial_values_dict(N)

    results = run_multiple_simulations(params, initial_values, end_time, no_runs)
    
    time = {} # Dict containing the times and how many extinctions happened at each time
    
    for result in results:
        if result['extinction'] == True:
            if result['extinction_time'] in time:
                time[result['extinction_time']] += 1
            else:
                time[result['extinction_time']] = 1

    return time

In [ ]:
# TEST plot extinctions time series for different values of N
beta = 1
gamma = 0.1
mu = 0.02/365
params = {'beta': beta, 'gamma': gamma, 'mu': mu}

N = [1e0, 1e1, 1e2, 1e3, 1e4, 1e5]

end_time = 150
no_runs = 100

for n in N:
    initial_values = get_initial_values_dict(n)
    extinction_times = get_number_of_extinctions_time_series(n, params, end_time, no_runs)

    lists = sorted(extinction_times.items()) # sorted by key, return a list of tuples

    x, y = zip(*lists) # unpack a list of pairs into two tuples
    
    y_cummulative = [y[0]]
    
    for i in range(len(y) - 1):
        y_cummulative.append(y_cummulative[i] + y[i+1])
    
    plt.plot(x, y_cummulative, label = rf'N = {n}')

plt.axis([0, end_time, 0, no_runs])
plt.xlabel('Time in days')
plt.ylabel('Total number of extinctions')
plt.title('The total # of extinctions over time for various population sizes N')
plt.legend()
plt.show()

In [ ]:
# TEST plot extinctions time series for different values of R_0 and N
betas = [0.1, 0.5, 1]
gamma = 0.1
mu = 0.02/365

N = [1e2, 1e3, 1e4]

end_time = 150
no_runs = 100

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))

for i, beta in enumerate(betas):
    params = {'beta': beta, 'gamma': gamma, 'mu': mu}
    for j, n in enumerate(N):
        initial_values = get_initial_values_dict(n)
        extinction_times = get_number_of_extinctions_time_series(n, params, end_time, no_runs)
    
        lists = sorted(extinction_times.items()) # sorted by key, return a list of tuples
    
        times, no_extinctions = zip(*lists) # unpack a list of pairs into two tuples
        
        no_extinctions_cummulative = [no_extinctions[0]]
        
        for k in range(len(no_extinctions) - 1):
            no_extinctions_cummulative.append(no_extinctions_cummulative[k] + no_extinctions[k+1])

        axes[i, j].plot(times, no_extinctions_cummulative)
        axes[i, j].set_title(f"N = {n}, R_0 = {beta/gamma}")
        axes[i, j].axis([0, end_time, 0, no_runs])

plt.tight_layout()
plt.show()

In [ ]:
# TEST plot extinctions time series for different values of R_0 and N
betas = [0.1, 0.5, 1]
gamma = 0.1
mu = 0.02/365

N = [1e2, 1e3, 1e4]

end_time = 150
no_runs = 100

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 6))

for i, n in enumerate(N):
    initial_values = get_initial_values_dict(n)
    for beta in betas:
        params = {'beta': beta, 'gamma': gamma, 'mu': mu}
        extinction_times = get_number_of_extinctions_time_series(n, params, end_time, no_runs)
    
        lists = sorted(extinction_times.items()) # sorted by key, return a list of tuples
    
        times, no_extinctions = zip(*lists) # unpack a list of pairs into two tuples
        
        no_extinctions_cummulative = [no_extinctions[0]]
        
        for k in range(len(no_extinctions) - 1):
            no_extinctions_cummulative.append(no_extinctions_cummulative[k] + no_extinctions[k+1])

        axes[i].plot(times, no_extinctions_cummulative, label = f'R_0 = {beta/gamma}')
        axes[i].set_title(f'N = {n}')
        axes[i].axis([0, end_time, 0, no_runs])
        axes[i].legend()

plt.tight_layout()
plt.show()

# Problem 2: Spatial Models - Networks

In this question you are asked to develop a set of experiments to design and evaluate
vaccination strategies using a network model. Using the package NDLib2 you should assess
the spread of a disease (SIR) across different types of model networks (Barabasi Albert,
Watts-Strogatz, Erdos-Reyni). Finally, you will run a simulated vaccination campaign
on a real contact network collected by sociopatterns (link). A modified version of this
dataset can be downloaded from Canvas, the network has been converted to a static
(non-temporal) form and some edges and nodes have been filtered out.

## 2.1 Implement SIR and Simulate

Implement SIR disease spread on the network. Think about your experimental design
and which parameters of the model you will want to vary. Design code that will allow
you run multiple simulations while varying the disease parameter.

### Functions to generate network, model, and run SIR simulations

In [ ]:
def create_network(network_type, num_nodes, seed = None, p = 0.01, m = 5, k = 6):
    """
    Creates a network for given type.

    Parameters:
        network_type (str): Type of network to generate. Options: 'Erdos-Renyi', 'Barabasi-Albert', 'Watts-Strogatz'.
        num_nodes (int): Number of nodes in the network.
        seed (int, optional): Seed for random number generation, can be used for reproducibility.
        p (float, optional): Probability of edge creation (for Erdos-Renyi) or reconnecting (Watts-Strogatz).
        m (int, optional): Number of edges to attach from a new node to existing nodes (for Barabasi-Albert).
        k (int, optional): Each node is connected to k nearest neighbors in ring topology (for Watts-Strogatz).
    
    Returns:
        networkx.Graph: Generated network graph based on specified type and parameters.
    
    Raises:
        ValueError: If the provided network type is not one of the available options.
    """
    if network_type == 'Erdos-Renyi':
        return nx.erdos_renyi_graph(num_nodes, p, seed)
    elif network_type == 'Barabasi-Albert':
        return nx.barabasi_albert_graph(num_nodes, m, seed)
    elif network_type == 'Watts-Strogatz':
        return nx.watts_strogatz_graph(num_nodes, k, p, seed)
    else:
        raise ValueError("Invalid network type. Choose from 'Erdos-Renyi', 'Barabasi-Albert', or 'Watts-Strogatz'.")

In [ ]:
def generate_model(network, beta, gamma, initial_infected_fraction = None, initial_infected_nodes = None):
    """
    Generates a SIR model for a given network and parameters.

    Parameters:
        network (networkx.Graph): Network on which the SIR model will run.
        beta (float): Transmission rate.
        gamma (float): Recovery rate.
        initial_infected_fraction (float, optional): Fraction of population initially infected.
        initial_infected_nodes (list, optional): Specific nodes to initially infect if initial_infected_fraction is not set.
    
    Returns:
        ndlib.models.epidemics.SIRModel: Configured SIR model ready for simulation.
    """
    # Create model
    model = ep.SIRModel(network)
    config = mc.Configuration()
    
    # Add model Parameters 
    config.add_model_parameter('beta', beta) 
    config.add_model_parameter('gamma', gamma)  

    # Set initial infected population (either by fraction or specific nodes)
    if initial_infected_fraction:
        config.add_model_parameter('percentage_infected', initial_infected_fraction)
    elif initial_infected_nodes is not None:
        config.add_model_initial_configuration('Infected', initial_infected_nodes)
    
    model.set_initial_status(config)

    return model

In [ ]:
def run_sir_simulations(network, beta, gamma, initial_infected_fraction = None, initial_infected_nodes = None, iterations = 100, num_simulations = 50):
    """
    Runs multiple SIR simulations on the given network and returns results.

    Parameters:
        network (networkx.Graph): Network on which the SIR model will run.
        beta (float): Transmission rate.
        gamma (float): Recovery rate.
        initial_infected_fraction (float, optional): Fraction of population initially infected.
        initial_infected_nodes (list, optional): Specific nodes to initially infect if initial_infected_fraction is not set.
        iterations (int): Number of time steps in each simulation.
        num_simulations (int): Number of simulation runs.
    
    Returns:
        list: List of results from multiple simulations.
    
    Raises:
        ValueError: If neither initial_infected_fraction nor initial_infected_nodes is provided.
    """
    # Generate the model
    model = generate_model(network, beta, gamma, initial_infected_fraction, initial_infected_nodes)

    # Run the model and retund results
    if initial_infected_fraction:
        return multi_runs(model, execution_number = num_simulations, iteration_number = iterations, nprocesses=4)
    elif initial_infected_nodes is not None:
        infection_sets = [initial_infected_nodes for _ in range(num_simulations)]
        return multi_runs(model, execution_number=num_simulations, iteration_number=iterations, infection_sets=infection_sets, nprocesses=4)
    else:
        raise ValueError("You need to provide either 'initial_infected_fraction' or 'initial_infected_nodes' parameter.")

## 2.2 Generate Networks of equivalent form

Using NetworkX generate multiple model networks with similar characteristics, again
think about the parameters associated with each network generator (e.g., Number of
nodes, connection probability,etc). Pick some network statistics (e.g., centrality measures,
degree distributions, etc.) that are interesting to measure in terms of spreading on the
network. You should generate multiple instances of each network type and then plot the
network statistics (you chose) and discuss how these statistic differ between network types
and for different parameter settings. You will use these generated networks in your SIR
experiments in the next part.

### Functions to measure basic network statistics

In [ ]:
def get_diameter(network):
    """
    Calculates the diameter of the network.
    
    Parameters:
        network (networkx.Graph): Network graph for which to calculate the diameter.
    
    Returns:
        int: Diameter of the network graph.
    """
    return nx.diameter(network)

In [ ]:
def get_average_path_length(network):
    """
    Calculates the average shortest path length of the network.
    
    Parameters:
        network (networkx.Graph): Network graph for which to calculate the average shortest path length.
    
    Returns:
        float: Average shortest path length.
    """
    return nx.average_shortest_path_length(network)

In [ ]:
def get_average_clustering(network):
    """
    Calculates the average clustering coefficient.
    
    Parameters:
        network (networkx.Graph): Network graph for which to calculate the average clustering coefficient.
    
    Returns:
        float: Average clustering coefficient.
    """
    return nx.average_clustering(network)

In [ ]:
def get_average_degree(network):
    """
    Calculates the average degree.
    
    Parameters:
        network (networkx.Graph): Network graph for which to calculate the average degree.
    
    Returns:
        float: Average degree.
    """
    return 2 * network.number_of_edges() / network.number_of_nodes()

In [ ]:
def show_degree_histogram(network, title = None):
    degree_histogram = nx.degree_histogram(network)

    degrees = range(len(degree_histogram))
    
    plt.bar(degrees, degree_histogram, color='lightgreen', edgecolor='black')
    
    plt.xlabel('Degree')
    plt.ylabel('Frequency')
    if title:
        plt.title(title)
    else:
        plt.title('Degree histogram')
    
    plt.show()

In [ ]:
def show_degree_probability(network, title = None):
    degree_histogram = nx.degree_histogram(network)

    num_nodes = network.number_of_nodes()

    degree_probability = [frequency / num_nodes for frequency in degree_histogram]

    # Scatter plot of the probability
    plt.scatter(range(1, len(degree_histogram) + 1), degree_probability)

    plt.xlabel('Degree k')
    plt.ylabel('Probability p(k)')
    if title:
        plt.title(title)
    else:
        plt.title('Degree distribution')

    plt.show()

### Centrality measures

In [ ]:
def get_average_degree_normalised(network):
    """
    Calculates the normalised version of average degree.
    
    Parameters:
        network (networkx.Graph): Network graph for which to calculate the normalised average degree.
    
    Returns:
        float: Normalised average degree.
    """
    degree_centrality = nx.degree_centrality(network)
    
    return sum(c_d / num_nodes for c_d in degree_centrality.values())

In [ ]:
def show_degree_centrality(network):
    """
    Plot a histogram of the degree centrality distribution for the network.
        
    Parameters:
    network (networkx.Graph): The network graph.
    
    Returns:
    None: Displays a histogram of degree centrality.
    """
    degree_centrality = nx.degree_centrality(network)
    
    # Plot a histogram of degree centrality values
    plt.hist(list(degree_centrality.values()), bins=10, color='lightgreen', edgecolor='black')
    plt.title('Degree Centrality Distribution')
    plt.xlabel('Degree Centrality (normalised)')
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
def show_betweenness_centrality(network):
    """
    Plot a histogram of the betweenness centrality distribution for the network.
        
    Parameters:
    network (networkx.Graph): The network graph.
    
    Returns:
    None: Displays a histogram of betweenness centrality.
    """
    betweenness_centrality = nx.betweenness_centrality(network)
    
    # Plot a histogram of betweenness centrality values
    plt.hist(list(betweenness_centrality.values()), bins=20, color='lightgreen', edgecolor='black')
    plt.title('Betweenness Centrality Distribution')
    plt.xlabel('Betweenness Centrality')
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
def show_closeness_centrality(network):
    """
    Plot a histogram of the closeness centrality distribution for the network.
        
    Parameters:
    network (networkx.Graph): The network graph.
    
    Returns:
    None: Displays a histogram of closeness centrality.
    """
    closeness_centrality = nx.closeness_centrality(network)
    
    # Plot a histogram of betweenness centrality values
    plt.hist(list(closeness_centrality.values()), bins=20, color='lightgreen', edgecolor='black')
    plt.title('Closeness Centrality Distribution')
    plt.xlabel('Closeness Centrality')
    plt.ylabel('Frequency')
    plt.show()

### Generate multiple networks and show statistics

In [ ]:
def print_basic_network_statistics(network):
    """
    Print fundamental statistics of the network.
        
    Parameters:
    network (networkx.Graph): The network graph.
    
    Returns:
    None: Prints diameter, average path length, average degree, clustering coefficient, and edge count.
    """
    print('Diameter =', get_diameter(network))
    print('Average path length =', get_average_path_length(network))
    print('Average degree =', get_average_degree(network))
    print('Clustering coefficient =', get_average_clustering(network))
    print('Number of edges =', network.number_of_edges())
    print('------------------------------------')

In [ ]:
def generate_multiple_networks(num_networks, network_type, num_nodes, seed, p = None, m = None, k = None):
    """
    Generate multiple networks of the same type and parameters with different seed
        
    Parameters:
    num_networks (int): The number of networks to be generated.
    network (networkx.Graph): The network graph.
    network_type (str): Type of network to generate. Options: 'Erdos-Renyi', 'Barabasi-Albert', 'Watts-Strogatz'.
    num_nodes (int): Number of nodes in the network.
    seed (int): Seed for random number generation, we will use it as the first seed and then increment.
    p (float, optional): Probability of edge creation (for Erdos-Renyi) or reconnecting (Watts-Strogatz).
    m (int, optional): Number of edges to attach from a new node to existing nodes (for Barabasi-Albert).
    k (int, optional): Each node is connected to k nearest neighbors in ring topology (for Watts-Strogatz).

    Returns:
    list of networkx.Graph: List of generated network graph based on specified type and parameters.
    """
    networks = []
    
    for n in range(num_networks):
        # Generate network
        network = create_network(network_type, num_nodes, seed = seed, p = p, k = k, m = m)
        networks.append(network)

        # Update seed
        seed += 1

    return networks

In [ ]:
def plot_network_histograms(network, title, save = False):
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    
    # Plot degree histogram
    degree_histogram = nx.degree_histogram(network)
    degrees = range(len(degree_histogram))
    
    axs[0].bar(degrees, degree_histogram, color='darkolivegreen', edgecolor='black')
    axs[0].set_xlabel('Degree')
    axs[0].set_ylabel('Frequency')
    # axs[0].set_title('Degree Histogram')

    # Plot degree centrality
    degree_centrality = nx.degree_centrality(network)
    
    axs[1].hist(list(degree_centrality.values()), bins=10, color='olive', edgecolor='black')
    axs[1].set_xlabel('Degree Centrality (normalised)')
    axs[1].set_ylabel('Frequency')
    # axs[1].set_title('Degree Centrality Distribution')
    
    # Plot betweenness centrality
    betweenness_centrality = nx.betweenness_centrality(network)
    
    axs[2].hist(list(betweenness_centrality.values()), bins=20, color='goldenrod', edgecolor='black')
    axs[2].set_xlabel('Betweenness Centrality')
    axs[2].set_ylabel('Frequency')
    # axs[2].set_title('Betweenness Centrality Distribution')
    
    # Plot closeness centrality
    closeness_centrality = nx.closeness_centrality(network)
    
    axs[3].hist(list(closeness_centrality.values()), bins=20, color='palegoldenrod', edgecolor='black')
    axs[3].set_xlabel('Closeness Centrality')
    axs[3].set_ylabel('Frequency')
    # axs[3].set_title('Closeness Centrality Distribution')

    plt.tight_layout()
    plt.suptitle(title, fontsize=18)
    plt.subplots_adjust(top = 0.9)

    if save:
        plt.savefig('network_histograms.pdf', dpi = 150)
    plt.show()

### Comparison of networks

#### Less connected networks

In [ ]:
# Generate networks
seed = 10
num_nodes = 1000

# Erdos-Renyi
er_network_1 = create_network('Erdos-Renyi', num_nodes, seed = seed, p = 0.01)

# Barabasi-Albert
ba_network_1 = create_network('Barabasi-Albert', num_nodes, seed = seed, m = 5)

# Watts-Strogatz
ws_network_1 = create_network('Watts-Strogatz', num_nodes, seed = seed, k = 10, p = 0.001)

# Print basic statistics
print('Erdos-Renyi')
print_basic_network_statistics(er_network_1)
print('Barabasi-Albert')
print_basic_network_statistics(ba_network_1)
print('Watts-Strogatz')
print_basic_network_statistics(ws_network_1)

# Plot histograms
plot_network_histograms(er_network_1, 'Degree and centrality distribution for Erdos-Renyi graph with N = 1000', save = False)
plot_network_histograms(ba_network_1, 'Degree and centrality distribution for Barabasi-Albert graph with N = 1000', save = False)
plot_network_histograms(ws_network_1, 'Degree and centrality distribution for Watts-Strogatz graph with N = 1000', save = False)

#### Heavily connected networks

In [ ]:
# Generate networks
seed = 10
num_nodes = 1000

# Erdos-Renyi
er_network_2 = create_network('Erdos-Renyi', num_nodes, seed = seed, p = 0.1)

# Barabasi-Albert
ba_network_2 = create_network('Barabasi-Albert', num_nodes, seed = seed, m = 50)

# Watts-Strogatz
ws_network_2 = create_network('Watts-Strogatz', num_nodes, seed = seed, k = 100, p = 0.001)

# Show statistics
print('Erdos-Renyi')
print_basic_network_statistics(er_network_2)
print('Barabasi-Albert')
print_basic_network_statistics(ba_network_2)
print('Watts-Strogatz')
print_basic_network_statistics(ws_network_2)

# Plot histograms
plot_network_histograms(er_network_2, 'Degree and centrality distribution for Erdos-Renyi graph with N = 1000')
plot_network_histograms(ba_network_2, 'Degree and centrality distribution for Barabasi-Albert graph with N = 1000')
plot_network_histograms(ws_network_2, 'Degree and centrality distribution for Watts-Strogatz graph with N = 1000')

#### Small networks

In [ ]:
# Generate networks
seed = 10
num_nodes = 100

# Erdos-Renyi
er_network_3 = create_network('Erdos-Renyi', num_nodes, seed = seed, p = 0.07)

# Barabasi-Albert
ba_network_3 = create_network('Barabasi-Albert', num_nodes, seed = seed, m = 3)

# Watts-Strogatz
ws_network_3 = create_network('Watts-Strogatz', num_nodes, seed = seed, k = 6, p = 0.001)

# Show statistics
print('Erdos-Renyi')
print_basic_network_statistics(er_network_3)
print('Barabasi-Albert')
print_basic_network_statistics(ba_network_3)
print('Watts-Strogatz')
print_basic_network_statistics(ws_network_3)

# Plot histograms
plot_network_histograms(er_network_3, 'Degree and centrality distribution for Erdos-Renyi graph with N = 1000')
plot_network_histograms(ba_network_3, 'Degree and centrality distribution for Barabasi-Albert graph with N = 1000')
plot_network_histograms(ws_network_3, 'Degree and centrality distribution for Watts-Strogatz graph with N = 1000')

## 2.3 Simulate SIR spread on the network

Simulate epidemic spreading on the networks you generated in the previous section
(NOTE: the simulations will be stochastic so think about random seeds and repitions).
You can vary the fraction of initial infected, which nodes are initially infected and other
disease parameters. Compare and discuss how the disease spreads in the different networks
under different conditions.

In [ ]:
def get_centrality_nodes(network, centrality_type, fraction=0.01, top=True):
    centrality = None
    if centrality_type == 'degree':
        centrality = nx.degree_centrality(network)
    elif centrality_type == 'betweenness':
        centrality = nx.betweenness_centrality(network)
    elif centrality_type == 'closeness':
        centrality = nx.closeness_centrality(network)
    else:
        raise ValueError("Invalid centrality type. Choose from 'degree', 'betweenness', or 'closeness'.")

    sorted_nodes = sorted(centrality.items(), key=lambda item: item[1], reverse=top)
    num_infected = int(fraction * len(network.nodes))
    
    return [node for node, _ in sorted_nodes[:num_infected]]

In [ ]:
def plot_multiple_trends(trends):
    # Plot the first one and include labels
    first_result = trends[0]['trends']['node_count']
    num_iterations = np.arange(len(first_result[0]))
    
    plt.plot(iterations, first_result[0], color='blue', alpha=0.01, label = 'Susceptible')
    plt.plot(iterations, first_result[1], color='red', alpha=0.01, label = 'Infected')
    plt.plot(iterations, first_result[2], color='green', alpha=0.01, label = 'Recovered')

    # Plot the rest (we assume there is more than 1 result)
    for result in trends[1:]:
        data = result['trends']['node_count']
        
        plt.plot(iterations, data[0], color='blue', alpha=0.01)
        plt.plot(iterations, data[1], color='red', alpha=0.01)
        plt.plot(iterations, data[2], color='green', alpha=0.01)

In [ ]:
def plot_mean_trends(trends, linestyle, labels, plot = plt):
    # Get the number of simulations and iterations
    num_simulations = len(trends)
    num_iterations = len(trends[0]['trends']['node_count'][0])

    # Initialize arrays for trends data
    susceptible = np.zeros((num_simulations, num_iterations))
    infected = np.zeros((num_simulations, num_iterations))
    recovered = np.zeros((num_simulations, num_iterations))

    # Add the trends row by row
    for i, result in enumerate(trends):
        data = result['trends']['node_count']
        susceptible[i, :] = data[0]
        infected[i, :] = data[1]
        recovered[i, :] = data[2]

    # Calculate the mean
    mean_susceptible = np.mean(susceptible, axis=0)
    mean_infected = np.mean(infected, axis=0)
    mean_recovered = np.mean(recovered, axis=0)

    # Get the x axis
    iterations = np.arange(num_iterations)

    # Plot the mean trends
    plot.plot(iterations, mean_susceptible, color = 'red', linestyle=linestyle, label = labels[0])
    plot.plot(iterations, mean_infected, color = 'blue', linestyle=linestyle, label = labels[1])
    plot.plot(iterations, mean_recovered, color = 'green', linestyle=linestyle, label = labels[2])

    plot.legend()

#### Compare different number of initially infected nodes

In [ ]:
def plot_sir_different_init_infected_number(er_network, ba_network, ws_network, beta, gamma, iterations, num_simulations, N, k, save = False):
    # Initialize a 1x3 grid of plots
    fig, axs = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    initially_infected_numbers = [1, 5, 10]
    
    # Loop through each option for initially infected nodes
    for idx, initially_infected in enumerate(initially_infected_numbers):
        # Generate initial infected nodes for this configuration
        initial_infected_nodes = np.random.choice(list(er_network.nodes()), initially_infected, replace=False)
    
        # Run simulations for each network type with current initial infected nodes
        er_trends = run_sir_simulations(er_network, beta, gamma, initial_infected_nodes=initial_infected_nodes, iterations=iterations, num_simulations=num_simulations)
        ba_trends = run_sir_simulations(ba_network, beta, gamma, initial_infected_nodes=initial_infected_nodes, iterations=iterations, num_simulations=num_simulations)
        ws_trends = run_sir_simulations(ws_network, beta, gamma, initial_infected_nodes=initial_infected_nodes, iterations=iterations, num_simulations=num_simulations)
    
        # Plot each trend on the corresponding subplot
        plot_mean_trends(er_trends, '-', ['Susceptible (ER)', 'Infected (ER)', 'Recovered (ER)'], axs[idx])
        plot_mean_trends(ba_trends, '--', ['Susceptible (BA)', 'Infected (BA)', 'Recovered (BA)'], axs[idx])
        plot_mean_trends(ws_trends, ':', ['Susceptible (WS)', 'Infected (WS)', 'Recovered (WS)'], axs[idx])
    
        # Set titles and labels for the subplot
        axs[idx].set_title(f'# of initially infected nodes = {initially_infected}')
        axs[idx].set_xlabel('Iterations')
        axs[idx].grid(True)
        if idx == 0:
            axs[idx].set_ylabel('# Individuals in Each Compartment')
    
    # Overall title
    plt.suptitle(f'Mean of SIR simulations for different network types with N = {N} and <k> = {k} and various numbers of initially infected nodess', fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit titles
    if save:
        plt.savefig(f'various_init_nodes_{N}_{k}.pdf', dpi = 150)
    plt.show()

In [ ]:
# Plot for all variants of the networks
betas = [1/10, 1/100, 1/6]
gamma = 1/10
iterations = 100
num_simulations = 30

# plot_sir_different_init_infected_number(er_network_1, ba_network_1, ws_network_1, betas[0], gamma, iterations, num_simulations, N = 1000, k = 10, save = True)
# plot_sir_different_init_infected_number(er_network_2, ba_network_2, ws_network_2, betas[1], gamma, iterations, num_simulations, N = 1000, k = 100, save = True)
# plot_sir_different_init_infected_number(er_network_3, ba_network_3, ws_network_3, betas[2], gamma, iterations, num_simulations, N = 100, k = 6, save = True)

#### Compare different value of spreading rate lambda

In [ ]:
def plot_sir_different_lambda(er_network, ba_network, ws_network, betas, iterations, num_simulations, N, k, save = False):
    # Initialize a 1x3 grid of plots
    fig, axs = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    # Generate 5 initial infected nodes
    initial_infected_nodes = np.random.choice(list(er_network.nodes()), 5, replace=False)

    gamma = 1/10
    
    # Loop through each option for initially infected nodes
    for idx, beta in enumerate(betas):
        # Run simulations for each network type with current value of beta
        er_trends = run_sir_simulations(er_network, beta, gamma, initial_infected_nodes=initial_infected_nodes, iterations=iterations, num_simulations=num_simulations)
        ba_trends = run_sir_simulations(ba_network, beta, gamma, initial_infected_nodes=initial_infected_nodes, iterations=iterations, num_simulations=num_simulations)
        ws_trends = run_sir_simulations(ws_network, beta, gamma, initial_infected_nodes=initial_infected_nodes, iterations=iterations, num_simulations=num_simulations)
    
        # Plot each trend on the corresponding subplot
        plot_mean_trends(er_trends, '-', ['Susceptible (ER)', 'Infected (ER)', 'Recovered (ER)'], axs[idx])
        plot_mean_trends(ba_trends, '--', ['Susceptible (BA)', 'Infected (BA)', 'Recovered (BA)'], axs[idx])
        plot_mean_trends(ws_trends, ':', ['Susceptible (WS)', 'Infected (WS)', 'Recovered (WS)'], axs[idx])
    
        # Set titles and labels for the subplot
        axs[idx].set_title(rf'$\beta$ = {beta:.3f}, $\gamma$ = {gamma}, $\lambda$ = {beta/gamma:.3f}')
        axs[idx].set_xlabel('Iterations')
        axs[idx].grid(True)
        if idx == 0:
            axs[idx].set_ylabel('# Individuals in Each Compartment')
    
    # Overall title
    plt.suptitle(rf'Mean of SIR simulations for different network types with N = {N} and <k> = {k} and various $\lambda$', fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit titles
    if save:
        plt.savefig(f'various_lambda_{N}_{k}_v2.pdf', dpi = 150)
    plt.show()

In [ ]:
# Plot for all variants of the networks
iterations = 100
num_simulations = 30
betas_1 = [1/20, 1/10, 1/5]
betas_2 = [1/200, 1/100, 1/50]
betas_3 = [1/12, 1/6, 1/3]

# plot_sir_different_lambda(er_network_1, ba_network_1, ws_network_1, betas_1, iterations, num_simulations, N = 1000, k = 10, save = True)
# plot_sir_different_lambda(er_network_2, ba_network_2, ws_network_2, betas_2, iterations, num_simulations, N = 1000, k = 100, save = True)
# plot_sir_different_lambda(er_network_3, ba_network_3, ws_network_3, betas_3, iterations, num_simulations, N = 100, k = 6, save = True)

#### Compare different choice of initially infected nodes by centrality

In [ ]:
def plot_sir_different_init_centrality(er_network, ba_network, ws_network, beta, gamma, iterations, num_simulations, N, k, save = False):
    # Initialize a 1x3 grid of plots
    fig, axs = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    centrality_types = ['degree', 'betweenness', 'closeness']
    
    # Loop through each option for initially infected nodes
    for idx, centrality_type in enumerate(centrality_types):
        # Generate initial infected nodes for this configuration
        initial_infected_nodes_er = get_centrality_nodes(er_network, centrality_type, fraction=5/N, top=True)
        initial_infected_nodes_ba = get_centrality_nodes(ba_network, centrality_type, fraction=5/N, top=True)
        initial_infected_nodes_ws = get_centrality_nodes(ws_network, centrality_type, fraction=5/N, top=True)
    
        # Run simulations for each network type with current initial infected nodes
        er_trends = run_sir_simulations(er_network, beta, gamma, initial_infected_nodes=initial_infected_nodes_er, iterations=iterations, num_simulations=num_simulations)
        ba_trends = run_sir_simulations(ba_network, beta, gamma, initial_infected_nodes=initial_infected_nodes_ba, iterations=iterations, num_simulations=num_simulations)
        ws_trends = run_sir_simulations(ws_network, beta, gamma, initial_infected_nodes=initial_infected_nodes_ws, iterations=iterations, num_simulations=num_simulations)
    
        # Plot each trend on the corresponding subplot
        plot_mean_trends(er_trends, '-', ['Susceptible (ER)', 'Infected (ER)', 'Recovered (ER)'], axs[idx])
        plot_mean_trends(ba_trends, '--', ['Susceptible (BA)', 'Infected (BA)', 'Recovered (BA)'], axs[idx])
        plot_mean_trends(ws_trends, ':', ['Susceptible (WS)', 'Infected (WS)', 'Recovered (WS)'], axs[idx])
    
        # Set titles and labels for the subplot
        axs[idx].set_title(f'Centrality type of infected nodes = {centrality_type}')
        axs[idx].set_xlabel('Iterations')
        axs[idx].grid(True)
        if idx == 0:
            axs[idx].set_ylabel('# Individuals in Each Compartment')
    
    # Overall title
    plt.suptitle(f'Mean of SIR simulations for different network types with N = {N} and <k> = {k} and different infected nodes by centrality', fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit titles
    if save:
        plt.savefig(f'various_init_nodes_centrality_{N}_{k}.pdf', dpi = 150)
    plt.show()

In [ ]:
# Plot for all variants of the networks
betas = [1/10, 1/100, 1/6]
gamma = 1/10
iterations = 100
num_simulations = 30

plot_sir_different_init_centrality(er_network_1, ba_network_1, ws_network_1, betas[0], gamma, iterations, num_simulations, N = 1000, k = 10, save = True)
plot_sir_different_init_centrality(er_network_2, ba_network_2, ws_network_2, betas[1], gamma, iterations, num_simulations, N = 1000, k = 100, save = True)
plot_sir_different_init_centrality(er_network_3, ba_network_3, ws_network_3, betas[2], gamma, iterations, num_simulations, N = 100, k = 6, save = True)

#### Compare different choice of initially infected nodes by degree centrality (least vs most)

In [ ]:
def plot_sir_different_init_degree_centrality(er_network, ba_network, ws_network, beta, gamma, iterations, num_simulations, N, k, save = False):
    # Initialize a 1x3 grid of plots
    fig, axs = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    # Get the initially infected nodes
    # 1. 5 nodes with the smallest degree_centrality
    # 2. 5 randomly selected nodes
    # 3. 5 nodes with the largest degree_centrality
    centrality_type = 'degree'
    initial_infected_nodes_random = np.random.choice(list(er_network.nodes()), 5, replace=False)
    initial_infected_nodes_er = [get_centrality_nodes(er_network, centrality_type, fraction=5/N, top=False), initial_infected_nodes_random, get_centrality_nodes(er_network, centrality_type, fraction=5/N, top=True)]
    initial_infected_nodes_ba = [get_centrality_nodes(ba_network, centrality_type, fraction=5/N, top=False), initial_infected_nodes_random, get_centrality_nodes(ba_network, centrality_type, fraction=5/N, top=True)]
    initial_infected_nodes_ws = [get_centrality_nodes(ws_network, centrality_type, fraction=5/N, top=False), initial_infected_nodes_random, get_centrality_nodes(ws_network, centrality_type, fraction=5/N, top=True)]
    
    # Loop through each option for initially infected nodes
    titles = ['Infected nodes with the smallest degree centrality', 'Randomly selected infected nodes', 'Infected nodes with the highest degree centrality']
    for idx in range(3):    
        # Run simulations for each network type with current initial infected nodes
        er_trends = run_sir_simulations(er_network, beta, gamma, initial_infected_nodes=initial_infected_nodes_er[idx], iterations=iterations, num_simulations=num_simulations)
        ba_trends = run_sir_simulations(ba_network, beta, gamma, initial_infected_nodes=initial_infected_nodes_ba[idx], iterations=iterations, num_simulations=num_simulations)
        ws_trends = run_sir_simulations(ws_network, beta, gamma, initial_infected_nodes=initial_infected_nodes_ws[idx], iterations=iterations, num_simulations=num_simulations)
    
        # Plot each trend on the corresponding subplot
        plot_mean_trends(er_trends, '-', ['Susceptible (ER)', 'Infected (ER)', 'Recovered (ER)'], axs[idx])
        plot_mean_trends(ba_trends, '--', ['Susceptible (BA)', 'Infected (BA)', 'Recovered (BA)'], axs[idx])
        plot_mean_trends(ws_trends, ':', ['Susceptible (WS)', 'Infected (WS)', 'Recovered (WS)'], axs[idx])
    
        # Set titles and labels for the subplot
        axs[idx].set_title(titles[idx])
        axs[idx].set_xlabel('Iterations')
        axs[idx].grid(True)
        if idx == 0:
            axs[idx].set_ylabel('# Individuals in Each Compartment')
    
    # Overall title
    plt.suptitle(f'Mean of SIR simulations for different network types with N = {N} and <k> = {k} and different infected nodes by centrality', fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit titles
    if save:
        plt.savefig(f'various_init_nodes_degree_centrality_{N}_{k}.pdf', dpi = 150)
    plt.show()

In [ ]:
# Plot for all variants of the networks
betas = [1/10, 1/100, 1/6]
gamma = 1/10
iterations = 100
num_simulations = 30

# plot_sir_different_init_degree_centrality(er_network_1, ba_network_1, ws_network_1, betas[0], gamma, iterations, num_simulations, N = 1000, k = 10, save = True)
# plot_sir_different_init_degree_centrality(er_network_2, ba_network_2, ws_network_2, betas[1], gamma, iterations, num_simulations, N = 1000, k = 100, save = True)
# plot_sir_different_init_degree_centrality(er_network_3, ba_network_3, ws_network_3, betas[2], gamma, iterations, num_simulations, N = 100, k = 6, save = True)

## 2.4 Dynamic Vaccination Campaign

Finally, we will conduct vaccination experiments using the sociopatterns dataset, see
Figure 1. You should design some code to load in the sociopatterns dataset (this file on
canvas includes a simple edgelist and NDLib and NetworkX provide ways to import this).

Now consider a scenario in which a disease is spreading on this network. You should run
multiple experiments, but assume that the disease always starts with a random selection
of 5 nodes infected.

You are to design a dynamic vaccination strategy in which you have a testing budget and
a limited number of vaccinations available per iteration of the model. Assume that you
have 200 tests in total, you can use a maximum number of tests per iteration (this will
vary per experiment see below), you can of course use less. You can use the tests at any
point during the spread and you may repeat tests on a node as often as you like. You
can assume that you know the network structure, but you can only discover the disease
status of a node after a test. Vaccinations can only be applied to susceptible people and
that they immediately move people to the removed state. Finally, you might consider
situations where the tests are not always accurate, but instead have some probability
(which you can vary) of being accurate. You can also assume that people remain removed
until the end of the simulation (no waning immunity).
You should compare your strategy against a simple null strategy which randomly assigns
vaccinations, you should design a strategy that at least out performs the null strategy.
Compare the strategies with different vaccination budgets of [1, 3, 5 and 10] per timestep
and compare with different testing accuracy [0.5, 0.75, 1.0] . Finally, keep in mind that
the purpose of the assignment is not to design the best strategy, but to evaluate you
strategy in a systematic and scientific manner.

### Load edgelist and generate a network from it

In [ ]:
def generate_conference_network():
    # Load the CSV data into a matrix
    nodes_and_edges_matrix = np.genfromtxt('transmission_network.csv', delimiter = ';')
    
    # Get the nodes and edges
    nodes = nodes_and_edges_matrix[1:,0]
    edges_matrix = nodes_and_edges_matrix[1:,1:]
    
    num_nodes = len(nodes)
    nodes = [str(node) for node in nodes]
    
    # Initialize the network with all the nodes
    network = nx.Graph()
    network.add_nodes_from(nodes)
    
    # Add the edges to the network
    for row in range(num_nodes):
        for col in range(row + 1, num_nodes):
            if edges_matrix[row, col] != 0:
                network.add_edge(nodes[row], nodes[col])
                
    return network

### Get statistics

In [ ]:
# Generate network
network = generate_conference_network()

# Show statistics
print('Average degree =', get_average_degree(network))
print('Clustering coefficient =', get_average_clustering(network))
print('Number of nodes =', network.number_of_nodes())
print('Number of edges =', network.number_of_edges())

# Plot histograms
plot_network_histograms(network, 'Degree and centrality distribution')

### Run SIR simulation without vaccination

In [ ]:
# Set parameters
network = generate_conference_network()
beta = 1/7
gamma = 1/10
initial_infected_fraction = 5/num_nodes # we have 5 random infected nodes at the beginning

trends = run_sir_simulations(network, beta, gamma, initial_infected_fraction = initial_infected_fraction, iterations = 100, num_simulations = 30)

# Plot
plot_mean_trends(trends, linestyle = '-', labels = ['Susceptible', 'Infected', 'Recovered'], plot = plt)

plt.title('SIR simulation on network') # TODO: improve
plt.xlabel('Iterations') # TODO: improve
plt.ylabel('Population') # TODO: improve
plt.tight_layout()
plt.show()

### Null strategy

In [ ]:
def simulate_null_vaccination_strategy(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations = 100):
    model = generate_model(network, beta, gamma, initial_infected_nodes = initial_infected_nodes)

    # Initially, no node was vaccinated yet
    not_vaccinated_nodes = list(network.nodes())

    # Initialize variables
    susceptibles = [369] # 374 - 5 who were initially infected
    infecteds = [5]
    recovered = [0]
    extinction = False
    vaccinations_used = 0

    for iteration in range(max_iterations):
        # Run an iteration
        result = model.iteration()
    
        # Check that there are still some infecteds left
        if result['node_count'][1] < 1:
            extinction = True
            break

        # Add the result to the respective lists
        susceptibles.append(result['node_count'][0])
        infecteds.append(result['node_count'][1])
        recovered.append(result['node_count'][2])
    
        # Get the current status of the nodes
        status = model.status
    
        # Pick nodes that are supposed to be vaccinated randomly
        if len(not_vaccinated_nodes) < vaccination_budget:
            nodes_to_vaccinate = not_vaccinated_nodes
        else:
            nodes_to_vaccinate = np.random.choice(not_vaccinated_nodes, vaccination_budget, replace = False)
        
        # Vaccinate by changing the status (vaccination only works when not infected, so the only chage is from S -> R)
        for node in nodes_to_vaccinate:
            # Remove node from the list of not vaccinated nodes
            not_vaccinated_nodes.remove(node)
            # We vaccinate every node maximum once. Even if we do not test, either the node was sucsceptible and it goes to R, or it was alredy in R or in I and once it recovers it will go to R.
            if status[node] == 0:
                model.status[node] = 2

        vaccinations_used += len(nodes_to_vaccinate)
    
    return {'susceptibles': susceptibles, 'infecteds': infecteds, 'recovered': recovered, 'extinction': extinction, 'vaccinations_used': vaccinations_used, 'number_of_iterations': iteration + 1}

In [ ]:
# Test one run of the null strategy
network = generate_conference_network()
beta = 1/7
gamma = 1/10
vaccination_budget = 5
initial_infected_nodes = np.random.choice(list(network.nodes()), vaccination_budget, replace=False)
max_iterations = 100

simulation_result = simulate_null_vaccination_strategy(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations)

In [ ]:
# Run 50 simulations of the null vaccination strategy for the same 5 initially infected nodes
# Generate network and set parameters
network = generate_conference_network()
beta = 1/7
gamma = 1/10
vaccination_budget = 5
initial_infected_nodes = np.random.choice(list(network.nodes()), vaccination_budget, replace=False)
max_iterations = 100

results = []

for simulation in range(50):
    results.append(simulate_null_vaccination_strategy(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))

In [ ]:
# Investigate the results
peak_number_of_infecteds = []
vaccinations_used = []
number_of_iterations = []

for result in results:
    peak_number_of_infecteds.append(max(result['infecteds']))
    vaccinations_used.append(result['vaccinations_used'])
    number_of_iterations.append(result['number_of_iterations'])

print('Average peak of number of infecteds:', np.mean(peak_number_of_infecteds))
print('Average number of vaccinations used:', np.mean(vaccinations_used))
print('Average number of iterations until extinction:', np.mean(number_of_iterations))

### Our vaccination strategy -- centrality

In [ ]:
def simulate_centrality_vaccination_strategy(network, centrality_type, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations = 100):
    # Generate model
    model = generate_model(network, beta, gamma, initial_infected_nodes = initial_infected_nodes)

    # Initialize variables
    susceptibles = [369] # 374 - 5 who were initially infected
    infecteds = [5]
    recovered = [0]
    extinction = False
    vaccinations_used = 0
    tests_used = 0

    # Get the centrality for all nodes and sort it
    if centrality_type == 'degree':
        centrality = nx.degree_centrality(network)
    elif centrality_type == 'betweenness':
        centrality = nx.betweenness_centrality(network)
    elif centrality_type == 'closeness':
        centrality = nx.closeness_centrality(network)
    else:
        raise ValueError('Invalid centrality type. Choose from "degree", "betweenness", or "closeness".')

    sorted_nodes = sorted(centrality.items(), key = lambda item: item[1])

    for iteration in range(max_iterations):
        # Get the current status of all nodes
        status = model.status

        # Test the nodes with highest degree centrality and vaccinate until you use all the vaccinations
        vaccinations_used_current_iter = 0
        while vaccinations_used_current_iter < vaccination_budget:
            # Check if there are any possible nodes left and get the node with highest degree centrality
            # TODO: clean up a bit
            if len(sorted_nodes) == 0:
                return {'susceptibles': susceptibles, 'infecteds': infecteds, 'recovered': recovered, 'extinction': extinction, 'vaccinations_used': vaccinations_used, 'number_of_iterations': iteration, 'tests_used': tests_used}

            node = sorted_nodes.pop()
            
            # If we still have tests left, test first and then vaccinate, if not, vaccinate anyway
            if tests_used < 200:
                # Check if the node is susceptible and if yes, vaccinate it, but do not vaccinate otherwise
                if status[node[0]] == 0:
                    model.status[node[0]] = 2
                    vaccinations_used_current_iter += 1
                tests_used += 1
            else:
                # Vaccinate, no matter the status (but effectivelly, the status will only change if susceptible)
                if status[node[0]] == 0:
                    model.status[node[0]] = 2
                vaccinations_used_current_iter += 1
    
        vaccinations_used += vaccinations_used_current_iter
            
        # Run an iteration
        result = model.iteration()
    
        # Check that there are still some infecteds left
        if result['node_count'][1] < 1:
            extinction = True
            break
    
        # Add the result to the respective lists
        susceptibles.append(result['node_count'][0])
        infecteds.append(result['node_count'][1])
        recovered.append(result['node_count'][2])

    return {'susceptibles': susceptibles, 'infecteds': infecteds, 'recovered': recovered, 'extinction': extinction, 'vaccinations_used': vaccinations_used, 'number_of_iterations': iteration + 1, 'tests_used': tests_used}

### Compare multiple strategies

In [ ]:
def get_all_strategies_results(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations):
    null_strategy_results = []
    degree_centrality_strategy_results = []
    betweenness_centrality_strategy_results = []
    closenesss_centrality_strategy_results = []

    for simulation in range(number_of_simulations):
        null_strategy_results.append(simulate_null_vaccination_strategy(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))
        degree_centrality_strategy_results.append(simulate_centrality_vaccination_strategy(network, 'degree', beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))
        betweenness_centrality_strategy_results.append(simulate_centrality_vaccination_strategy(network, 'betweenness', beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))
        closenesss_centrality_strategy_results.append(simulate_centrality_vaccination_strategy(network, 'closeness', beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))

    return [null_strategy_results, degree_centrality_strategy_results, betweenness_centrality_strategy_results, closenesss_centrality_strategy_results]

In [ ]:
def show_result_stats(results):
    peak_number_of_infecteds = []
    vaccinations_used = []
    number_of_iterations = []
    
    for result in results:
        peak_number_of_infecteds.append(max(result['infecteds']))
        vaccinations_used.append(result['vaccinations_used'])
        number_of_iterations.append(result['number_of_iterations'])
    
    print('Average peak of number of infecteds:', np.mean(peak_number_of_infecteds))
    print('Average number of vaccinations used:', np.mean(vaccinations_used))
    print('Average number of iterations until extinction:', np.mean(number_of_iterations))

In [ ]:
# Get the strategies results
network = generate_conference_network()
beta = 1/7
gamma = 1/10
vaccination_budget = 5
max_iterations = 100
number_of_simulations = 30
initial_nodes_variations = 10

results = [[],[],[],[]]

# Test for different initially infected nodes
for _ in range(initial_nodes_variations):
    initial_infected_nodes = np.random.choice(list(network.nodes()), 5, replace=False)
    
    result = get_all_strategies_results(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

    results[0] = results[0] + result[0]
    results[1] = results[1] + result[1]
    results[2] = results[2] + result[2]
    results[3] = results[3] + result[3]

In [ ]:
# Print the results
print('Lambda =', beta/gamma)

# Get epidemic treshold
degree_sequence = np.array([d for _, d in network.degree()])
second_moment = np.mean(degree_sequence**2)
average_degree = get_average_degree(network)
epidemic_treshold = average_degree/second_moment
print('Epidemic treshold:', epidemic_treshold)
print('---------------------------------------')
print('Null strategy results:')
show_result_stats(results[0])
print('---------------------------------------')
print('Degree centrality strategy results:')
show_result_stats(results[1])
print('---------------------------------------')
print('Betweenness centrality strategy results:')
show_result_stats(results[2])
print('---------------------------------------')
print('Closenesss centrality strategy results:')
show_result_stats(results[3])

In [ ]:
# Get the strategies results
network = generate_conference_network()
beta = 1/100
gamma = 1/10
vaccination_budget = 5
max_iterations = 100
number_of_simulations = 30
initial_nodes_variations = 3

results = [[],[],[],[]]

# Test for different initially infected nodes
for _ in range(initial_nodes_variations):
    initial_infected_nodes = np.random.choice(list(network.nodes()), 5, replace=False)
    
    result = get_all_strategies_results(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

    results[0] = results[0] + result[0]
    results[1] = results[1] + result[1]
    results[2] = results[2] + result[2]
    results[3] = results[3] + result[3]

In [ ]:
# Print the results
print('Lambda =', beta/gamma)

# Get epidemic treshold
degree_sequence = np.array([d for _, d in network.degree()])
second_moment = np.mean(degree_sequence**2)
average_degree = get_average_degree(network)
epidemic_treshold = average_degree/second_moment
print('Epidemic treshold:', epidemic_treshold)
print('---------------------------------------')
print('Null strategy results:')
show_result_stats(results[0])
print('---------------------------------------')
print('Degree centrality strategy results:')
show_result_stats(results[1])
print('---------------------------------------')
print('Betweenness centrality strategy results:')
show_result_stats(results[2])
print('---------------------------------------')
print('Closenesss centrality strategy results:')
show_result_stats(results[3])

In [ ]:
def get_result_stats(results):
    peak_number_of_infecteds = []
    vaccinations_used = []
    number_of_iterations = []
    
    for result in results:
        peak_number_of_infecteds.append(max(result['infecteds']))
        vaccinations_used.append(result['vaccinations_used'])
        number_of_iterations.append(result['number_of_iterations'])
    
    return np.mean(peak_number_of_infecteds), np.mean(vaccinations_used), np.mean(number_of_iterations)

In [ ]:
# beta_values = [1/100, 1/50, 1/25]
# gamma = 1/10
# vaccination_budgets = [1, 3, 5, 10]
# max_iterations = 100
# number_of_simulations = 30
# initial_infected_nodes = np.random.choice(list(network.nodes()), 5, replace=False)

# # Store results for each (beta, vaccination_budget) combination
# results_summary = []

# for beta in beta_values:
#     for vaccination_budget in vaccination_budgets:
#         # Initialize results for this combination
#         results = [[], [], [], []]

#         # Test for different initially infected nodes
#         for _ in range(1):
#             result = get_all_strategies_results(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

#             results[0] += result[0]
#             results[1] += result[1]
#             results[2] += result[2]
#             results[3] += result[3]

#         # Calculate stats for each strategy
#         null_stats = get_result_stats(results[0])
#         degree_stats = get_result_stats(results[1])
#         betweenness_stats = get_result_stats(results[2])
#         closeness_stats = get_result_stats(results[3])

#         # Store summary of results
#         results_summary.append({
#             'beta': beta,
#             'vaccination_budget': vaccination_budget,
#             'null_strategy': null_stats,
#             'degree_centrality': degree_stats,
#             'betweenness_centrality': betweenness_stats,
#             'closeness_centrality': closeness_stats
#         })

# # Print summary of results
# for result in results_summary:
#     print(f"Beta: {result['beta']}, Vaccination Budget: {result['vaccination_budget']}")
#     print(f"Null Strategy - Peak: {result['null_strategy'][0]}, Vaccinations: {result['null_strategy'][1]}, Iterations: {result['null_strategy'][2]}")
#     print(f"Degree Centrality - Peak: {result['degree_centrality'][0]}, Vaccinations: {result['degree_centrality'][1]}, Iterations: {result['degree_centrality'][2]}")
#     print(f"Betweenness Centrality - Peak: {result['betweenness_centrality'][0]}, Vaccinations: {result['betweenness_centrality'][1]}, Iterations: {result['betweenness_centrality'][2]}")
#     print(f"Closeness Centrality - Peak: {result['closeness_centrality'][0]}, Vaccinations: {result['closeness_centrality'][1]}, Iterations: {result['closeness_centrality'][2]}")
#     print("\n")

In [ ]:
def simulate_centrality_vaccination_strategy_test_accurancy(test_accurancy, network, centrality_type, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations = 100):
    # Generate model
    model = generate_model(network, beta, gamma, initial_infected_nodes = initial_infected_nodes)

    # Initialize variables
    susceptibles = [369] # 374 - 5 who were initially infected
    infecteds = [5]
    recovered = [0]
    extinction = False
    vaccinations_used = 0
    tests_used = 0

    # Get the centrality for all nodes and sort it
    if centrality_type == 'degree':
        centrality = nx.degree_centrality(network)
    elif centrality_type == 'betweenness':
        centrality = nx.betweenness_centrality(network)
    elif centrality_type == 'closeness':
        centrality = nx.closeness_centrality(network)
    else:
        raise ValueError('Invalid centrality type. Choose from "degree", "betweenness", or "closeness".')

    sorted_nodes = sorted(centrality.items(), key = lambda item: item[1])

    for iteration in range(max_iterations):
        # Get the current status of all nodes
        status = model.status

        # Test the nodes with highest degree centrality and vaccinate until you use all the vaccinations
        vaccinations_used_current_iter = 0
        while vaccinations_used_current_iter < vaccination_budget:
            # Check if there are any possible nodes left and get the node with highest degree centrality
            # TODO: clean up a bit
            if len(sorted_nodes) == 0:
                return {'susceptibles': susceptibles, 'infecteds': infecteds, 'recovered': recovered, 'extinction': extinction, 'vaccinations_used': vaccinations_used, 'number_of_iterations': iteration, 'tests_used': tests_used}

            node = sorted_nodes.pop()
            
            # If we still have tests left, test first and then vaccinate, if not, vaccinate anyway
            if tests_used < 200:
                # Pick randomly if the test will be accurate or not for given test accurancy
                accurate = (np.random.rand() < test_accurancy)

                if accurate:
                    tested_node_status = status[node[0]]
                else:
                    status_options = [0,1,2]
                    status_options.remove(status[node[0]])
                    tested_node_status = np.random.choice(status_options)
                
                # Check if the node is susceptible and if yes, vaccinate it, but do not vaccinate otherwise
                if tested_node_status == 0:
                    if status[node[0]] == 0:
                        model.status[node[0]] = 2
                    vaccinations_used_current_iter += 1
                tests_used += 1
            else:
                # Vaccinate, no matter the status (but effectivelly, the status will only change if susceptible)
                if status[node[0]] == 0:
                    model.status[node[0]] = 2
                vaccinations_used_current_iter += 1
    
        vaccinations_used += vaccinations_used_current_iter
            
        # Run an iteration
        result = model.iteration()
    
        # Check that there are still some infecteds left
        if result['node_count'][1] < 1:
            extinction = True
            break
    
        # Add the result to the respective lists
        susceptibles.append(result['node_count'][0])
        infecteds.append(result['node_count'][1])
        recovered.append(result['node_count'][2])

    return {'susceptibles': susceptibles, 'infecteds': infecteds, 'recovered': recovered, 'extinction': extinction, 'vaccinations_used': vaccinations_used, 'number_of_iterations': iteration + 1, 'tests_used': tests_used}

In [ ]:
def get_all_strategies_results_test_accurancy(test_accurancy, network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations):
    null_strategy_results = []
    degree_centrality_strategy_results = []
    betweenness_centrality_strategy_results = []
    closenesss_centrality_strategy_results = []

    for simulation in range(number_of_simulations):
        null_strategy_results.append(simulate_null_vaccination_strategy(network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))
        degree_centrality_strategy_results.append(simulate_centrality_vaccination_strategy_test_accurancy(test_accurancy, network, 'degree', beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))
        betweenness_centrality_strategy_results.append(simulate_centrality_vaccination_strategy_test_accurancy(test_accurancy, network, 'betweenness', beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))
        closenesss_centrality_strategy_results.append(simulate_centrality_vaccination_strategy_test_accurancy(test_accurancy, network, 'closeness', beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations))

    return [null_strategy_results, degree_centrality_strategy_results, betweenness_centrality_strategy_results, closenesss_centrality_strategy_results]

In [ ]:
# Test different testing accurancies
network = generate_conference_network()
beta = 1/30
gamma = 1/10
vaccination_budget = 5
max_iterations = 80
number_of_simulations = 30
initial_nodes_variations = 1
test_accurancy = 0.2

results = [[],[],[],[]]

# Test for different initially infected nodes
for _ in range(initial_nodes_variations):
    initial_infected_nodes = np.random.choice(list(network.nodes()), 5, replace=False)
    
    result = get_all_strategies_results_test_accurancy(test_accurancy, network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

    results[0] = results[0] + result[0]
    results[1] = results[1] + result[1]
    results[2] = results[2] + result[2]
    results[3] = results[3] + result[3]

In [ ]:
# Print the results
print('Lambda =', beta/gamma)

# Get epidemic treshold
degree_sequence = np.array([d for _, d in network.degree()])
second_moment = np.mean(degree_sequence**2)
average_degree = get_average_degree(network)
epidemic_treshold = average_degree/second_moment
print('Epidemic treshold:', epidemic_treshold)
print('---------------------------------------')
print('Null strategy results:')
show_result_stats(results[0])
print('---------------------------------------')
print('Degree centrality strategy results:')
show_result_stats(results[1])
print('---------------------------------------')
print('Betweenness centrality strategy results:')
show_result_stats(results[2])
print('---------------------------------------')
print('Closenesss centrality strategy results:')
show_result_stats(results[3])

In [ ]:
# Test different testing accurancies
network = generate_conference_network()
beta = 1/30
gamma = 1/10
vaccination_budget = 5
max_iterations = 80
number_of_simulations = 30

initial_infected_nodes = np.random.choice(list(network.nodes()), 5, replace=False)

# Test accurancy 0.5
test_accurancy = 0.5
results_1 = [[],[],[],[]]

result = get_all_strategies_results_test_accurancy(test_accurancy, network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

results_1[0] = result[0]
results_1[1] = result[1]
results_1[2] = result[2]
results_1[3] = result[3]

# Test accurancy 0.7
test_accurancy = 0.7
results_2 = [[],[],[],[]]

result = get_all_strategies_results_test_accurancy(test_accurancy, network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

results_2[0] = result[0]
results_2[1] = result[1]
results_2[2] = result[2]
results_2[3] = result[3]

# Test accurancy 0.9
test_accurancy = 0.9
results_3 = [[],[],[],[]]

result = get_all_strategies_results_test_accurancy(test_accurancy, network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

results_3[0] = result[0]
results_3[1] = result[1]
results_3[2] = result[2]
results_3[3] = result[3]

# Test accurancy 1
test_accurancy = 1
results_4 = [[],[],[],[]]

result = get_all_strategies_results_test_accurancy(test_accurancy, network, beta, gamma, initial_infected_nodes, vaccination_budget, max_iterations, number_of_simulations)

results_4[0] = result[0]
results_4[1] = result[1]
results_4[2] = result[2]
results_4[3] = result[3]

In [ ]:
# Print the results
print('TESTING ACCURACY: 0.5')
print('Null strategy results:')
show_result_stats(results_1[0])
print('---------------------------------------')
print('Degree centrality strategy results:')
show_result_stats(results_1[1])
print('---------------------------------------')
print('Betweenness centrality strategy results:')
show_result_stats(results_1[2])
print('---------------------------------------')
print('Closenesss centrality strategy results:')
show_result_stats(results_1[3])

print('---------------------------------------')
print('---------------------------------------')

print('TESTING ACCURACY: 0.7')
print('Null strategy results:')
show_result_stats(results_2[0])
print('---------------------------------------')
print('Degree centrality strategy results:')
show_result_stats(results_2[1])
print('---------------------------------------')
print('Betweenness centrality strategy results:')
show_result_stats(results_2[2])
print('---------------------------------------')
print('Closenesss centrality strategy results:')
show_result_stats(results_2[3])

print('---------------------------------------')
print('---------------------------------------')

print('TESTING ACCURACY: 0.9')
print('Null strategy results:')
show_result_stats(results_3[0])
print('---------------------------------------')
print('Degree centrality strategy results:')
show_result_stats(results_3[1])
print('---------------------------------------')
print('Betweenness centrality strategy results:')
show_result_stats(results_3[2])
print('---------------------------------------')
print('Closenesss centrality strategy results:')
show_result_stats(results_3[3])

In [ ]:
# TODO: this could be made in an iteration
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 6))

# Plot infecteds peak for null strategy and different testing accuracy
# Accuracy = 0.5
num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_1[0])
axes[0].plot(0.5, num_infecteds, 'ro', label = 'Null strategy')
axes[1].plot(0.5, vaccinations_used, 'ro', label = 'Null strategy')
axes[2].plot(0.5, num_iterations, 'ro', label = 'Null strategy')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_1[1])
axes[0].plot(0.5, num_infecteds, 'bo', label = 'Degree centrality')
axes[1].plot(0.5, vaccinations_used, 'bo', label = 'Degree centrality')
axes[2].plot(0.5, num_iterations, 'bo', label = 'Degree centrality')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_1[2])
axes[0].plot(0.5, num_infecteds, 'go', label = 'Betweenness centrality')
axes[1].plot(0.5, vaccinations_used, 'go', label = 'Betweenness centrality')
axes[2].plot(0.5, num_iterations, 'go', label = 'Betweenness centrality')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_1[3])
axes[0].plot(0.5, num_infecteds, 'yo', label = 'Closenesss centrality')
axes[1].plot(0.5, vaccinations_used, 'yo', label = 'Closenesss centrality')
axes[2].plot(0.5, num_iterations, 'yo', label = 'Closenesss centrality')


# Accuracy = 0.7
num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_2[0])
axes[0].plot(0.7, num_infecteds, 'ro')
axes[1].plot(0.7, vaccinations_used, 'ro')
axes[2].plot(0.7, num_iterations, 'ro')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_2[1])
axes[0].plot(0.7, num_infecteds, 'bo')
axes[1].plot(0.7, vaccinations_used, 'bo')
axes[2].plot(0.7, num_iterations, 'bo')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_2[2])
axes[0].plot(0.7, num_infecteds, 'go')
axes[1].plot(0.7, vaccinations_used, 'go')
axes[2].plot(0.7, num_iterations, 'go')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_2[3])
axes[0].plot(0.7, num_infecteds, 'yo')
axes[1].plot(0.7, vaccinations_used, 'yo')
axes[2].plot(0.7, num_iterations, 'yo')


# Accuracy = 0.9
num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_3[0])
axes[0].plot(0.9, num_infecteds, 'ro')
axes[1].plot(0.9, vaccinations_used, 'ro')
axes[2].plot(0.9, num_iterations, 'ro')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_3[1])
axes[0].plot(0.9, num_infecteds, 'bo')
axes[1].plot(0.9, vaccinations_used, 'bo')
axes[2].plot(0.9, num_iterations, 'bo')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_3[2])
axes[0].plot(0.9, num_infecteds, 'go')
axes[1].plot(0.9, vaccinations_used, 'go')
axes[2].plot(0.9, num_iterations, 'go')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_3[3])
axes[0].plot(0.9, num_infecteds, 'yo')
axes[1].plot(0.9, vaccinations_used, 'yo')
axes[2].plot(0.9, num_iterations, 'yo')


# Accuracy = 1
num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_4[0])
axes[0].plot(1, num_infecteds, 'ro')
axes[1].plot(1, vaccinations_used, 'ro')
axes[2].plot(1, num_iterations, 'ro')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_4[1])
axes[0].plot(1, num_infecteds, 'bo')
axes[1].plot(1, vaccinations_used, 'bo')
axes[2].plot(1, num_iterations, 'bo')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_4[2])
axes[0].plot(1, num_infecteds, 'go')
axes[1].plot(1, vaccinations_used, 'go')
axes[2].plot(1, num_iterations, 'go')

num_infecteds, vaccinations_used, num_iterations = get_result_stats(results_4[3])
axes[0].plot(1, num_infecteds, 'yo')
axes[1].plot(1, vaccinations_used, 'yo')
axes[2].plot(1, num_iterations, 'yo')


axes[0].set_title('Average peak of number of infecteds')
axes[1].set_title('Average number of vaccinations used')
axes[2].set_title('Average number of iterations until extinction')

axes[0].set_xlabel('Testing accuracy')
axes[1].set_xlabel('Testing accuracy')
axes[2].set_xlabel('Testing accuracy')

axes[0].legend()
axes[1].legend()
axes[2].legend()

axes[0].grid()
axes[1].grid()
axes[2].grid()

plt.savefig('Testing accuracy.pdf', dpi=150)
plt.show()